# Headless OAuth2 Refresh-Token Grant — REST Table Gateway

**Design spec · Approach B · 2026-06-03**
**Design epic:** `bd-1mmd` · **Status:** draft (awaiting decision confirmation)
**Component:** `crates/spur-notebook/rest-table-gateway`

> **Goal.** Turn the gateway's fake "OAuth2" (paste-a-static-bearer-token) into a *real* grant: the user supplies `client_id` + `client_secret` + a `refresh_token` once, and the gateway mints and auto-refreshes short-lived access tokens against the provider's `token_url` — fully headless, no browser. The authorization-code/browser flow (Approach C) is **deferred** and designed to layer on top of this core.

## Brief lock

| Field | Value |
|---|---|
| Surface | Backend runtime + connection wizard UX |
| Audience | SPUR notebook users connecting SaaS APIs (Notion, Slack, Google, Salesforce…) |
| Integration target | `ManifestAdapter::scan` async auth resolution + new `oauth.rs` token service |
| Non-goal | Browser consent, callback server, PKCE, OS-keychain storage, OAUTH2_CC, TWO_STEP/Plaid |

## Proposed defaults — please confirm (the 3 open decisions)

- **(a) Credentials = env vars**, reusing the wizard's existing collection flow: `<NAME>_CLIENT_ID`, `<NAME>_CLIENT_SECRET`, `<NAME>_REFRESH_TOKEN`.
- **(b) Token cache = in-memory only** (process-global), no disk persistence across restarts.
- **(c) Proactive refresh** ~60 s before `expires_at`.

These three are marked **〚confirm〛** wherever they appear below.

## 1 · The gap, precisely

Today every Nango `OAUTH2` provider collapses to a **static bearer token**:

- `auth_cfg` (nango.rs) maps any non-API_KEY/non-BASIC mode to `AuthCfg::Bearer { env: "<NAME>_TOKEN" }`.
- `ManifestAdapter::resolve_auth` (manifest_adapter.rs:40) is **synchronous** and just does `std::env::var(env)` — no exchange, no expiry, no refresh.
- The runtime enum is `ResolvedAuth = None | Bearer | Header | Basic | QueryParam`. There is no grant machinery anywhere.
- The provider snapshot is **stripped**: OAuth entries carry only `auth_mode` + `proxy.base_url` — no `token_url`.

**Consequence:** a user must hand-paste a bearer token that expires in ~1 hour, then re-paste it forever. That is not an OAuth integration.

### Current vs. target

| | Today | Target (Approach B) |
|---|---|---|
| Credential | one static `*_TOKEN` (expires) | `*_CLIENT_ID` + `*_CLIENT_SECRET` + `*_REFRESH_TOKEN` (durable) |
| Token lifetime | dies in ~1 h, manual re-paste | minted per need, auto-refreshed |
| `resolve_auth` | sync env read | async; token-service call |
| New runtime piece | — | `oauth.rs` token service + cache |
| `apply_auth` / `HttpFetch` | unchanged | **unchanged** (still `bearer_auth`) |

The single most important integration fact: **`resolve_auth()` is already called *inside* the async `scan`** (manifest_adapter.rs:200), immediately before `HttpFetch` is built. So the entire change is contained at that one async seam — nothing downstream of `apply_auth` moves.

## 2 · Integration architecture

Two planes meet at the manifest: a **config plane** (how an `Oauth2Refresh` manifest is produced + credentials collected) and a **request plane** (how a token is minted at scan time). New surfaces are highlighted.

```mermaid
flowchart TB
    subgraph CONFIG["Config plane — build the manifest & collect secrets"]
        SNAP["nango_providers_snapshot.yaml<br/>(+ token_url for 11 OAuth providers)"]:::changed
        PE["ProviderEntry<br/>(+ token_url field)"]:::changed
        AC["nango::auth_cfg<br/>OAUTH2 → Oauth2Refresh"]:::changed
        MAN["Manifest TOML<br/>auth = scheme oauth2_refresh"]:::changed
        WIZ["AddRestApiWizard (UI)<br/>collects 3 secrets"]:::changed
        ENV["env vars<br/>NAME_CLIENT_ID / _SECRET / _REFRESH_TOKEN"]:::confirm
        SNAP --> PE --> AC --> MAN
        WIZ -->|required_env_vars| ENV
        MAN -.declares required_env_vars.-> WIZ
    end

    subgraph REQUEST["Request plane — mint a token at scan time"]
        SCAN["ManifestAdapter::scan (async)"]
        RA["resolve_auth().await<br/>(now async)"]:::changed
        TS["oauth::TokenService.access_token()"]:::new
        CACHE["process-global token cache<br/>HashMap key→{token, expires_at}"]:::new
        HTTPX["POST token_url<br/>grant_type=refresh_token"]:::new
        RESB["ResolvedAuth::Bearer(token)"]
        HF["HttpFetch { auth }"]
        AA["apply_auth → req.bearer_auth"]
        API["Provider REST API"]
        SCAN --> RA --> TS
        TS -->|hit & fresh| CACHE
        TS -->|miss / expiring| HTTPX --> CACHE
        CACHE --> RESB
        RA --> RESB --> HF --> AA --> API
    end

    MAN ==>|loaded by| SCAN
    ENV ==>|read by| TS
    PROV["Provider token endpoint"]:::ext
    HTTPX --> PROV

    classDef new fill:#1f6f3f,stroke:#3fb950,color:#e6edf3;
    classDef changed fill:#3d2f12,stroke:#d29922,color:#e6edf3;
    classDef confirm fill:#1f3a5f,stroke:#58a6ff,color:#e6edf3;
    classDef ext fill:#21262d,stroke:#8b949e,color:#8b949e;
```

**Legend** — 🟩 new · 🟧 changed · 🟦 〚confirm〛 env-var creds · ⬛ external. **The blast radius stops at `resolve_auth`** — `HttpFetch`, `apply_auth`, the pagination/templating code, and the DuckDB vtab layer are all untouched.

## 3 · Request-time sequence — mint, cache, refresh

What happens on a `SELECT … FROM notion_pages()` once the connection is `ready`. The cache turns N queries into ≤1 token exchange per access-token lifetime.

```mermaid
sequenceDiagram
    autonumber
    participant K as DuckDB kernel
    participant S as ManifestAdapter::scan
    participant R as resolve_auth (async)
    participant T as oauth::TokenService
    participant C as token cache
    participant P as Provider token_url
    participant A as Provider REST API

    K->>S: scan(table)
    S->>R: await resolve_auth()
    Note over R: scheme == Oauth2Refresh
    R->>T: access_token(cfg, ctx)
    T->>C: lookup(key)
    alt cache hit & now < expires_at - 60s 〚confirm〛
        C-->>T: cached access_token
    else miss or within 60s of expiry
        T->>P: POST grant_type=refresh_token<br/>(client_id, client_secret, refresh_token, scope?)
        P-->>T: { access_token, expires_in }
        T->>C: store(key, token, now + expires_in)
        C-->>T: ok
    end
    T-->>R: access_token
    R-->>S: ResolvedAuth::Bearer(token)
    S->>A: GET base_url/path  (Authorization: Bearer …)
    A-->>S: rows
    S-->>K: RecordBatch
```

**Failure handling (spec):** if the refresh POST returns non-2xx or the body lacks `access_token`, `TokenService` returns a typed `GatewayError::Auth` that surfaces as a clear "token refresh failed for `<connection>`" error — never a silent unauthenticated request (today's `resolve_auth` silently degrades to `ResolvedAuth::None`; the new path must not). A `401` from the **REST API** with a fresh token invalidates the cache entry and retries the exchange exactly once.

## 4 · The new manifest contract

One new variant on the existing `AuthCfg` enum (`manifest.rs`) is the whole runtime contract. Everything else feeds or consumes it.

```rust
// manifest.rs — add to enum AuthCfg (serde tag = "scheme")
Oauth2Refresh {
    token_url: String,          // may contain ${connectionConfig.*}; run through resolve_template
    client_id_env: String,      // 〚confirm〛 default "<NAME>_CLIENT_ID"
    client_secret_env: String,  // 〚confirm〛 default "<NAME>_CLIENT_SECRET"
    refresh_token_env: String,  // 〚confirm〛 default "<NAME>_REFRESH_TOKEN"
    #[serde(default)]
    scope: Option<String>,
}
```

```toml
# Example generated manifest (Notion)
[source]
name = "notion"
base_url = "https://api.notion.com/v1"
auth = { scheme = "oauth2_refresh", token_url = "https://api.notion.com/v1/oauth/token", \
         client_id_env = "NOTION_CLIENT_ID", client_secret_env = "NOTION_CLIENT_SECRET", \
         refresh_token_env = "NOTION_REFRESH_TOKEN" }
```

## 5 · Module integration & build order

```mermaid
flowchart LR
    T1["T1 · manifest.rs<br/>AuthCfg::Oauth2Refresh<br/>+ TOML round-trip"]:::root
    T2["T2 · oauth.rs<br/>TokenService + cache<br/>(mock token endpoint test)"]:::root
    T3["T3 · manifest_adapter.rs<br/>async resolve_auth + scan wiring"]:::mid
    T4["T4 · nango.rs + snapshot<br/>emit oauth2_refresh, ProviderEntry.token_url"]:::leaf
    T5["T5 · mcp/mod.rs<br/>required_env_vars surfacing"]:::leaf

    T1 --> T3
    T2 --> T3
    T1 --> T4
    T1 --> T5

    classDef root fill:#1f6f3f,stroke:#3fb950,color:#e6edf3;
    classDef mid fill:#3d2f12,stroke:#d29922,color:#e6edf3;
    classDef leaf fill:#1f3a5f,stroke:#58a6ff,color:#e6edf3;
```

**Parallelism:** T1 ∥ T2 at the root; once T1 lands, T4 ∥ T5 run in parallel; T3 waits on T1 + T2. T3 is the only task with real async/locking nuance (suggest claude-code); the rest are codex-shaped single-file edits. T4 edits `nango.rs` — sequence it after the already-merged Gap 2/5 work to avoid same-file churn.

## 6 · UX — connection lifecycle

The wizard you already saw for `NOTION_TOKEN` extends to **three** secrets. The state machine the UI renders:

```mermaid
stateDiagram-v2
    [*] --> NotConfigured
    NotConfigured --> AwaitingCredentials: pick OAuth2 provider
    AwaitingCredentials --> AwaitingCredentials: missing_env_vars > 0
    AwaitingCredentials --> Ready: all 3 secrets present
    Ready --> Minting: first query / token expiring
    Minting --> Ready: access_token cached
    Minting --> AuthError: refresh POST fails
    AuthError --> Minting: user fixes secret / retry
    Ready --> [*]: remove connection

    note right of AwaitingCredentials
        wizard shows 3 fields:
        Client ID · Client Secret · Refresh Token
        (write-only, masked)
    end note
    note right of Minting
        invisible to user on success;
        a subtle "refreshing" pill only
        if it takes > 400ms
    end note
```

| State | What the user sees | Driven by |
|---|---|---|
| `AwaitingCredentials` | Amber banner + 3 empty masked fields | `missing_env_vars` from connection status |
| `Ready` | Green "ready" pill, table functions become callable | all `required_env_vars` set |
| `Minting` | Nothing, or a quiet "refreshing…" pill if slow | `TokenService` cache miss at scan |
| `AuthError` | Red inline error on the failing field + retry | typed `GatewayError::Auth` |

The next cell renders the **designed UX** for these states.

In [2]:
# open-design artifact — OAuth2 refresh-grant connection UX
from IPython.display import HTML

html = r"""
<div class="oauthux">
<style>
.oauthux{--bg:#0d1117;--p1:#161b22;--p2:#1c2128;--bd:#30363d;--tx:#e6edf3;--mu:#8b949e;--fa:#6e7681;
  --gn:#3fb950;--am:#d29922;--rd:#f85149;--bl:#58a6ff;--pu:#bc8cff;
  --mono:ui-monospace,SFMono-Regular,Menlo,monospace;--sans:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;
  background:var(--bg);color:var(--tx);font-family:var(--sans);padding:26px;border-radius:14px;
  max-width:980px;margin:0 auto;border:1px solid var(--bd);line-height:1.45;}
.oauthux *{box-sizing:border-box;}
.oauthux .hd{display:flex;align-items:center;justify-content:space-between;margin-bottom:4px;}
.oauthux h2{font-size:19px;margin:0;font-weight:650;letter-spacing:-.2px;}
.oauthux .sub{color:var(--mu);font-size:12.5px;margin:2px 0 20px;}
.oauthux .badge{font-family:var(--mono);font-size:11px;padding:4px 9px;border-radius:20px;border:1px solid;}
.oauthux .b-oauth{color:var(--pu);border-color:#5a3e8a;background:#241a33;}
.oauthux .grid{display:grid;grid-template-columns:1fr 1fr;gap:18px;}
.oauthux .card{background:var(--p1);border:1px solid var(--bd);border-radius:11px;padding:16px 17px;}
.oauthux .card h3{font-size:12px;text-transform:uppercase;letter-spacing:.6px;color:var(--mu);margin:0 0 13px;font-weight:600;}
/* fields */
.oauthux .field{margin-bottom:13px;}
.oauthux .field label{display:block;font-size:12px;color:var(--tx);margin-bottom:5px;font-weight:550;}
.oauthux .field .env{font-family:var(--mono);font-size:10.5px;color:var(--bl);}
.oauthux .inp{display:flex;align-items:center;gap:8px;background:var(--p2);border:1px solid var(--bd);
  border-radius:7px;padding:8px 10px;font-family:var(--mono);font-size:12px;color:var(--tx);}
.oauthux .inp.filled{border-color:#2f6f46;}
.oauthux .inp .dots{letter-spacing:2px;color:var(--fa);}
.oauthux .inp .tick{margin-left:auto;color:var(--gn);font-size:12px;}
.oauthux .hint{font-size:10.5px;color:var(--fa);margin-top:4px;}
/* stepper */
.oauthux .steps{display:flex;flex-direction:column;gap:0;}
.oauthux .step{display:flex;gap:11px;align-items:flex-start;position:relative;padding-bottom:15px;}
.oauthux .step:not(:last-child)::before{content:"";position:absolute;left:7px;top:18px;bottom:0;width:2px;background:var(--bd);}
.oauthux .dot{width:16px;height:16px;border-radius:50%;flex:none;margin-top:1px;border:2px solid var(--bd);background:var(--bg);z-index:1;}
.oauthux .dot.done{background:var(--gn);border-color:var(--gn);}
.oauthux .dot.cur{background:var(--bg);border-color:var(--gn);box-shadow:0 0 0 3px #1f4d31;}
.oauthux .dot.idle{background:var(--bg);}
.oauthux .step .t{font-size:13px;font-weight:600;}
.oauthux .step .d{font-size:11.5px;color:var(--mu);}
.oauthux .pill{font-family:var(--mono);font-size:10px;padding:2px 7px;border-radius:12px;margin-left:7px;vertical-align:middle;}
.oauthux .pill.gn{color:var(--gn);background:#10301d;}
.oauthux .pill.am{color:var(--am);background:#33280c;}
.oauthux .pill.rd{color:var(--rd);background:#3a1715;}
/* timeline */
.oauthux .tl{margin-top:16px;background:var(--p1);border:1px solid var(--bd);border-radius:11px;padding:16px 17px;}
.oauthux .bar{position:relative;height:34px;margin:18px 0 8px;border-radius:6px;
  background:linear-gradient(90deg,#10301d 0%,#10301d 78%,#33280c 78%,#33280c 100%);border:1px solid var(--bd);}
.oauthux .mk{position:absolute;top:-15px;font-family:var(--mono);font-size:9.5px;color:var(--mu);transform:translateX(-50%);}
.oauthux .mk::after{content:"";position:absolute;left:50%;top:14px;width:1px;height:34px;background:var(--fa);}
.oauthux .mk.refresh{color:var(--am);}
.oauthux .seg{position:absolute;top:9px;font-family:var(--mono);font-size:10px;color:var(--tx);}
.oauthux .axis{display:flex;justify-content:space-between;font-family:var(--mono);font-size:9.5px;color:var(--fa);}
/* request preview */
.oauthux .req{margin-top:16px;background:#0a0d12;border:1px solid var(--bd);border-radius:11px;padding:14px 16px;font-family:var(--mono);font-size:11.5px;}
.oauthux .req .c{color:var(--mu);}
.oauthux .req .k{color:var(--pu);}
.oauthux .req .s{color:var(--gn);}
.oauthux .req .dim{color:var(--fa);}
.oauthux .legend{margin-top:16px;font-size:11px;color:var(--mu);display:flex;gap:18px;flex-wrap:wrap;}
.oauthux .legend b{color:var(--tx);font-weight:600;}
</style>

  <div class="hd">
    <h2>Connect data source · Notion</h2>
    <span class="badge b-oauth">OAUTH2 · refresh-token grant</span>
  </div>
  <div class="sub">Provide your OAuth app credentials once. SPUR mints and refreshes access tokens for you — no token pasting, no expiry.</div>

  <div class="grid">
    <!-- LEFT: credential form -->
    <div class="card">
      <h3>Credentials &nbsp;·&nbsp; <span style="color:var(--gn)">3 / 3 set</span></h3>
      <div class="field">
        <label>Client ID <span class="env">NOTION_CLIENT_ID</span></label>
        <div class="inp filled"><span>5f9a1c2e-…-b7d4</span><span class="tick">&#10003;</span></div>
      </div>
      <div class="field">
        <label>Client Secret <span class="env">NOTION_CLIENT_SECRET</span></label>
        <div class="inp filled"><span class="dots">&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;</span><span class="tick">&#10003;</span></div>
      </div>
      <div class="field">
        <label>Refresh Token <span class="env">NOTION_REFRESH_TOKEN</span></label>
        <div class="inp filled"><span class="dots">&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;&bull;</span><span class="tick">&#10003;</span></div>
        <div class="hint">Obtained once, out-of-band (Approach C will capture this via a browser flow). Stored write-only as an env var.</div>
      </div>
    </div>

    <!-- RIGHT: lifecycle stepper -->
    <div class="card">
      <h3>Connection lifecycle</h3>
      <div class="steps">
        <div class="step"><span class="dot done"></span><div><div class="t">Awaiting credentials</div><div class="d">wizard blocks until all 3 secrets present</div></div></div>
        <div class="step"><span class="dot cur"></span><div><div class="t">Ready<span class="pill gn">now</span></div><div class="d">table functions callable · no token yet</div></div></div>
        <div class="step"><span class="dot idle"></span><div><div class="t">Minting<span class="pill am">on query</span></div><div class="d">cache miss → refresh-token POST</div></div></div>
        <div class="step"><span class="dot idle"></span><div><div class="t">Auth error<span class="pill rd">recoverable</span></div><div class="d">typed error on the failing field · retry</div></div></div>
      </div>
    </div>
  </div>

  <!-- token timeline -->
  <div class="tl">
    <h3 style="font-size:12px;text-transform:uppercase;letter-spacing:.6px;color:var(--mu);margin:0 0 6px;font-weight:600;">Access-token lifecycle &nbsp;·&nbsp; one exchange per token window, not per query</h3>
    <div class="bar">
      <span class="mk" style="left:0%">mint (t0)</span>
      <span class="mk refresh" style="left:78%">refresh&nbsp;@&nbsp;expiry&minus;60s</span>
      <span class="mk" style="left:100%">expires_in</span>
      <span class="seg" style="left:30%">queries served from cache &rarr;</span>
    </div>
    <div class="axis"><span>0s</span><span>cached &amp; fresh</span><span>3540s &#9889;</span><span>3600s</span></div>
  </div>

  <!-- request preview -->
  <div class="req">
    <div><span class="c"># cache miss → token service</span></div>
    <div><span class="k">POST</span> https://api.notion.com/v1/oauth/token</div>
    <div class="dim">grant_type=refresh_token &amp; client_id=… &amp; client_secret=… &amp; refresh_token=…</div>
    <div><span class="s">200</span> { "access_token": "…", "expires_in": 3600 }&nbsp;&nbsp;<span class="dim">→ cache.store(key, +3600s)</span></div>
    <div style="margin-top:8px"><span class="c"># every scan, unchanged path</span></div>
    <div><span class="k">GET</span> https://api.notion.com/v1/databases/…&nbsp;&nbsp;<span class="dim">Authorization: Bearer &lt;minted&gt;</span></div>
  </div>

  <div class="legend">
    <span><b style="color:var(--gn)">&#9679;</b> done / ready</span>
    <span><b style="color:var(--am)">&#9679;</b> minting / refreshing</span>
    <span><b style="color:var(--rd)">&#9679;</b> auth error (never silent)</span>
    <span><b style="color:var(--bl)">env</b> = 〚confirm〛 credential storage</span>
  </div>
</div>
"""
HTML(html)

## 7 · Acceptance, testing, scope

### Acceptance criteria
- A manifest with `auth = { scheme = "oauth2_refresh", … }` round-trips through TOML and parses to `AuthCfg::Oauth2Refresh`.
- At scan time, with the 3 env vars set, the gateway POSTs `grant_type=refresh_token` to `token_url`, applies the minted token as `Bearer`, and serves rows.
- A second scan within the token window performs **no** additional token POST (cache hit).
- A failed refresh surfaces a typed `GatewayError::Auth` — **never** a silent unauthenticated request.
- `nango::auth_cfg` emits `Oauth2Refresh` for `auth_mode: OAUTH2` when `token_url` is present; falls back to today's `Bearer` when absent (no regression).
- Connection status reports the 3 `required_env_vars`; wizard collects them via the existing `awaiting_credentials` flow.

### Testing strategy
- **T1** unit: TOML round-trip for the new variant (mirror `toml_roundtrips`).
- **T2** unit: `TokenService` against a `wiremock` token endpoint — cache miss exchanges, cache hit doesn't, expiry triggers re-exchange, non-2xx → `Auth` error. (Pattern: `manifest_adapter.rs::tests::base_url_templated`.)
- **T3** integration: `scan` end-to-end with mock token + mock API, asserting the `Authorization: Bearer` header carries the minted token.
- **T4** unit: `provider_to_manifest_stub` emits `Oauth2Refresh` for an OAUTH2 provider with `token_url`.

### Out of scope (→ Approach C epic)
Browser authorization-code consent, loopback callback server, PKCE + state/CSRF, obtaining the **initial** refresh token, OS-keychain / encrypted storage, OAUTH2_CC (client-credentials) and TWO_STEP/Plaid. Approach C reuses this token-service core unchanged — it only adds the front-half that *acquires* the refresh token that B then maintains.

### Open decisions 〚confirm〛
1. **(a)** Credential storage = env vars (vs. encrypted store now)?
2. **(b)** Token cache in-memory only (vs. persisted to `connections.json`/keychain)?
3. **(c)** Refresh skew = 60 s.
4. Cache key granularity: `(token_url, refresh_token_env)` vs. per-connection-name — affects multi-connection isolation.

---
*Next step after sign-off: `writing-plans` → beads-backed DAG (T1–T5) → `submit_plan`.*